In [1]:
import os
import time
import pandas as pd
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types

In [2]:
spark = (
    SparkSession.builder
        .appName("CryptoETL")
        .config("spark.master", "spark://spark-master:7077")
        # ---- Iceberg + Hive Catalog ----
        .config("spark.sql.catalog.hive_catalog", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.hive_catalog.catalog-impl", "org.apache.iceberg.hive.HiveCatalog")
        .config("spark.sql.catalog.hive_catalog.uri", "thrift://hive-metastore:9083")
        .config("spark.sql.catalog.hive_catalog.warehouse", "s3a://crypto-data-lake/")
        # ---- Default catalog
        .config("spark.sql.defaultCatalog", "hive_catalog")
        # ---- S3 (MinIO) ----
        .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
        .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
        .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        # ---- Iceberg Extensions ----
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
        .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
        # ---- Extra JARs ----
        .config("spark.jars", ",".join([
            "/opt/spark-extra-jars/iceberg-spark-runtime-3.5_2.12-1.6.1.jar",
            "/opt/spark-extra-jars/hadoop-aws-3.3.4.jar",
            "/opt/spark-extra-jars/aws-java-sdk-bundle-1.12.262.jar"
        ]))
        .getOrCreate()
)

25/10/07 06:05:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [4]:
spark.sql("""
select * from serving_db.klines
order by group_id
""").show()

+--------+-------------------+----------------+----------+----------+---------+-----------+--------+----------------+------------+-------+
|group_id|         group_date|       open_time|open_price|high_price|low_price|close_price|  volume|      close_time|landing_date| symbol|
+--------+-------------------+----------------+----------+----------+---------+-----------+--------+----------------+------------+-------+
| 1954368|2025-09-27 00:00:00|1758931200686229|    0.7918|    0.7918|   0.7897|     0.7901|465779.1|1758932095846561|  2025-09-27|ADAUSDT|
| 1954369|2025-09-27 00:15:00|1758932100583189|    0.7902|    0.7924|     0.79|     0.7911|163062.6|1758932997788221|  2025-09-27|ADAUSDT|
| 1954370|2025-09-27 00:30:00|1758933000808905|    0.7912|     0.792|   0.7905|     0.7913|202817.4|1758933895280530|  2025-09-27|ADAUSDT|
| 1954371|2025-09-27 00:45:00|1758933900464103|    0.7913|    0.7927|   0.7899|     0.7923|434212.2|1758934797111165|  2025-09-27|ADAUSDT|
| 1954372|2025-09-27 01:00:

In [3]:
spark.sql("""
select group_id, group_date, open_price, high_price, low_price, close_price, volume, ema7, ema20, trend, pattern, symbol from serving_db.pattern_two
where pattern is not null
""").show()

25/10/07 06:06:10 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

+--------+-------------------+----------+----------+---------+-----------+--------+------+------+---------+-----------------+-------+
|group_id|         group_date|open_price|high_price|low_price|close_price|  volume|  ema7| ema20|    trend|          pattern| symbol|
+--------+-------------------+----------+----------+---------+-----------+--------+------+------+---------+-----------------+-------+
| 1954413|2025-09-27 11:15:00|    0.7837|    0.7853|   0.7837|      0.785|121594.4|0.7835| 0.784|downtrend|bullish engulfing|ADAUSDT|
| 1954445|2025-09-27 19:15:00|    0.7803|    0.7823|     0.78|     0.7821| 95233.0|0.7808|0.7817|downtrend|bullish engulfing|ADAUSDT|
+--------+-------------------+----------+----------+---------+-----------+--------+------+------+---------+-----------------+-------+



In [20]:
df_sorted = (
    spark.sql("select * from serving_db.klines")
    .coalesce(1) # one partition, not shuffle
    .sortWithinPartitions("group_id")
)

In [21]:
df_sorted.show()

+--------+-------------------+----------------+----------+----------+---------+-----------+--------+----------------+
|group_id|         group_date|       open_time|open_price|high_price|low_price|close_price|  volume|      close_time|
+--------+-------------------+----------------+----------+----------+---------+-----------+--------+----------------+
| 1948896|2025-08-01 00:00:00|1754006400328945| 115764.07| 115829.46|115308.55|  115313.01| 302.159|1754007299467573|
| 1948897|2025-08-01 00:15:00|1754007300010950| 115313.01|  115933.0| 115313.0|  115800.01| 450.539|1754008199447993|
| 1948898|2025-08-01 00:30:00|1754008200077603|  115800.0|  115800.0|115423.87|  115517.98| 184.425|1754009099900832|
| 1948899|2025-08-01 00:45:00|1754009100223687| 115517.99| 115527.53|114313.13|  115427.27|1589.765|1754009999974074|
| 1948900|2025-08-01 01:00:00|1754010000363342| 115427.27| 115609.99| 114600.0|   114649.9| 681.883|1754010899995166|
| 1948901|2025-08-01 01:15:00|1754010900041356|  114649.

In [22]:
schema = types.StructType([
    *df_sorted.schema.fields,  # keep all original fields
    types.StructField("ema7", types.DoubleType(), True),
    types.StructField("ema20", types.DoubleType(), True)
])

In [23]:
def round_half_up(x, decimals=2):
    if x is None:
        return None
    factor = 10 ** decimals
    return float(int(x * factor + 0.5)) / factor

def calc_ema(value, state):
    if value is None:
        return None
    prev, buffer, period, k = state["prev"], state["buffer"], state["period"], state["k"]
    if prev is None:
        buffer.append(value)
        if len(buffer) == period:
            ema = sum(buffer) / len(buffer)
        else:
            ema = None
    else:
        ema = (value - prev) * k + prev

    state["prev"] = ema
    return ema

def rounded(dec):
    return float(dec.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP))

def ema_in_chunks(iterator):
    ema_configs = {
        "ema7": {"period": 7, "k": 2 / (7 + 1), "prev": None, "buffer": []},
        "ema20": {"period": 20, "k": 2 / (20 + 1), "prev": None, "buffer": []}
    }

    for pdf in iterator:
        ema7, ema20 = [], []
        for p in pdf["close_price"]:
            price = float(p)
            e7 = calc_ema(price, ema_configs["ema7"])
            ema7.append(round_half_up(e7, 2) if e7 is not None else None)
            e20 = calc_ema(price, ema_configs["ema20"])
            ema20.append(round_half_up(e20, 2) if e20 is not None else None)
            
        pdf["ema7"] = ema7
        pdf["ema20"] = ema20
        pdf = pdf[[*pdf.columns[:-2], "ema7", "ema20"]]
        yield pdf

In [24]:
df = df_sorted.mapInPandas(ema_in_chunks, schema)

In [25]:
df.createOrReplaceTempView("temp")

In [26]:
df = spark.sql("""
with cte as (
    select
        *,
        case 
            when ema7 > ema20 then 'uptrend' 
            when ema7 < ema20 then 'downtrend' 
            else NULL 
        end as trend,
        LAG(open_price, 1) over(order by group_id) as open_price_prev,
        LAG(close_price, 1) over(order by group_id) as close_price_prev
    from temp
)
select
    group_id,
    group_date,
    open_time,
    open_price,
    high_price,
    low_price,
    close_price,
    volume,
    close_time,
    ema7,
    ema20,
    trend,
    case 
        when close_price_prev < open_price_prev
            and close_price > open_price
            and open_price < close_price_prev
            and close_price > open_price_prev
            and trend = 'downtrend'
        then 'bullish engulfing'
        when close_price_prev > open_price_prev
            and close_price < open_price
            and open_price > close_price_prev
            and close_price < open_price_prev
            and trend = 'uptrend'
        then 'bearish engulfing'
        else NULL
    end as pattern
from cte
""")

In [27]:
df.show(100)

+--------+-------------------+----------------+----------+----------+---------+-----------+--------+----------------+---------+---------+---------+-----------------+
|group_id|         group_date|       open_time|open_price|high_price|low_price|close_price|  volume|      close_time|     ema7|    ema20|    trend|          pattern|
+--------+-------------------+----------------+----------+----------+---------+-----------+--------+----------------+---------+---------+---------+-----------------+
| 1948896|2025-08-01 00:00:00|1754006400328945| 115764.07| 115829.46|115308.55|  115313.01| 302.159|1754007299467573|     NULL|     NULL|     NULL|             NULL|
| 1948897|2025-08-01 00:15:00|1754007300010950| 115313.01|  115933.0| 115313.0|  115800.01| 450.539|1754008199447993|     NULL|     NULL|     NULL|             NULL|
| 1948898|2025-08-01 00:30:00|1754008200077603|  115800.0|  115800.0|115423.87|  115517.98| 184.425|1754009099900832|     NULL|     NULL|     NULL|             NULL|
| 19

25/09/29 01:05:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/09/29 01:05:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/09/29 01:05:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [28]:
spark.sql("""
drop table if exists serving_db.pattern_two
""")

DataFrame[]

In [29]:
df.writeTo("serving_db.pattern_two").tableProperty("format-version", "2").createOrReplace()

25/09/29 01:06:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/09/29 01:06:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/09/29 01:06:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [4]:
spark.sql("""
select * from serving_db.pattern_two
""").orderBy("group_id", ascending=True).show(100)

+--------+-------------------+----------------+----------+----------+---------+-----------+-------+----------------+---------+---------+---------+-----------------+
|group_id|         group_date|       open_time|open_price|high_price|low_price|close_price| volume|      close_time|     ema7|    ema20|    trend|          pattern|
+--------+-------------------+----------------+----------+----------+---------+-----------+-------+----------------+---------+---------+---------+-----------------+
| 1948896|2025-08-01 00:00:00|1754006400328945| 115764.07| 115829.46|115308.55|  115313.01| 302.16|1754007299467573|     NULL|     NULL|     NULL|             NULL|
| 1948897|2025-08-01 00:15:00|1754007300010950| 115313.01|  115933.0| 115313.0|  115800.01| 450.54|1754008199447993|     NULL|     NULL|     NULL|             NULL|
| 1948898|2025-08-01 00:30:00|1754008200077603|  115800.0|  115800.0|115423.87|  115517.98| 184.43|1754009099900832|     NULL|     NULL|     NULL|             NULL|
| 1948899|

In [3]:
spark.sql("""
select * from serving_db.pattern_two where pattern is not null
""").show()

25/10/04 13:39:43 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

+--------+-------------------+----------------+----------+----------+---------+-----------+------+----------------+---------+---------+---------+-----------------+
|group_id|         group_date|       open_time|open_price|high_price|low_price|close_price|volume|      close_time|     ema7|    ema20|    trend|          pattern|
+--------+-------------------+----------------+----------+----------+---------+-----------+------+----------------+---------+---------+---------+-----------------+
| 1948917|2025-08-01 05:15:00|1754025300156136| 115847.53| 115847.53|115481.14|  115588.93| 90.23|1754026199363498|115666.29| 115585.6|  uptrend|bearish engulfing|
| 1948952|2025-08-01 14:00:00|1754056800059538| 114352.04|  115264.2|113988.47|  115054.42|767.85|1754057699799492|115141.39|115205.81|downtrend|bullish engulfing|
+--------+-------------------+----------------+----------+----------+---------+-----------+------+----------------+---------+---------+---------+-----------------+



In [1]:
!jupyter nbconvert --to script transform_job_pattern_two.ipynb

[NbConvertApp] Converting notebook transform_job_pattern_two.ipynb to script
[NbConvertApp] Writing 5253 bytes to transform_job_pattern_two.py
